# 实验五：贪心算法

---

In [1]:
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>

## 题目1：贪心算法解决 TSP 问题

**实验目的**

1. 掌握贪心法的设计思想。
2. 理解旅行商问题（TSP）的求解过程，并用程序实现。
3. 熟练使用二维数组存储城市间的距离。

**实验内容**

给定 $n$ 个城市及任意两座城市之间的距离。从一座城市出发，每座城市恰好访问一次，最后回到出发城市，使回路的总距离最短。输出访问路线和总距离，并附文字说明。任选一种语言编写程序，并分析算法复杂度。

### 算法思路

用二维数组 `dist` 保存城市之间的距离，`dist[u][v]` 表示从城市 `u` 到城市 `v` 的距离。约定共有 `n ≥ 1` 座城市，编号为 `0` 到 `n - 1`，起点固定为 `0`，同一城市到自身的距离为 `0`。

最近邻法从起点出发，每一步只在未访问的城市中，选择**离当前城市最近**的一座。按编号从小到大扫描候选城市，只有距离严格更小时才替换当前候选，因此距离相同时选择编号较小的城市。

用 `visited` 标记已访问的城市，用 `route` 依次记录城市编号，用 `current` 和 `total` 分别记录当前位置和累计距离。开始时先记录并标记起点。每选出一座新城市，就把本段距离计入 `total`，将该城市写入路线并标记为已访问，然后更新 `current`。经过 `n - 1` 轮后，从最后一座城市返回起点，把回程距离也计入总距离。

每一轮恰好加入一座未访问的城市，因此路线的前 `n` 个位置包含全部城市且各出现一次。末尾再写入起点表示回程。最近邻法保证得到合法回路，但每次只比较眼前这一段的距离，所得回路可能比全局最短回路更长。

### 伪代码

输入为城市数 `n` 和 `n × n` 的距离矩阵 `dist`。下标从 `0` 开始，所有循环均包含所写的起止下标。`route` 共预留 `n + 1` 个位置，最后一个位置用于记录回到起点。

```text
NearestNeighbor(dist, n)
    start ← 0
    current ← start
    total ← 0
    创建长度为 n 的数组 visited，初值均为 false
    创建长度为 n + 1 的数组 route
    visited[start] ← true
    route[0] ← start

    对 i 从 1 到 n - 1
        minDist ← +∞

        对 v 从 0 到 n - 1
            如果 visited[v] = false 且 dist[current][v] < minDist
                next ← v
                minDist ← dist[current][v]

        route[i] ← next
        visited[next] ← true
        total ← total + minDist
        current ← next

    route[n] ← start
    total ← total + dist[current][start]

    输出“最近邻路线”，按顺序用箭头连接 route[0] 到 route[n]
    输出“路线总距离”和 total
```

每轮开始时至少还有一座未访问的城市，且城际距离为有限值，因此扫描结束时一定已选出 `next`。当 `n = 1` 时，选城循环执行零次，路线为 `0 → 0`，总距离为 `0`。

### 源代码实现

C 实现使用固定容量 `MAX_CITIES = 100`，要求 `1 ≤ n ≤ MAX_CITIES`。调用时矩阵列数为 `MAX_CITIES`，有效距离填在左上角的 `n × n` 区域，采用非负整数且主对角线为 `0`。

In [2]:
enum { MAX_CITIES = 100 };

void tsp_nearest_neighbor(const int dist[][MAX_CITIES], int n)
{
    if (n < 1 || n > MAX_CITIES) {
        printf("输入错误：城市数量必须在 1 到 %d 之间。\n", MAX_CITIES);
        return;
    }

    const int start = 0;
    int current = start;
    int visited[MAX_CITIES] = {0};
    int route[MAX_CITIES + 1];
    long long total = 0;

    visited[start] = 1;
    route[0] = start;

    for (int i = 1; i < n; ++i) {
        int next = -1;
        int minDist = INT_MAX;

        for (int v = 0; v < n; ++v) {
            if (visited[v]) {
                continue;
            }

            // 首个可行候选直接入选，其后只在距离严格更小时替换。
            if (next == -1 || dist[current][v] < minDist) {
                next = v;
                minDist = dist[current][v];
            }
        }

        // 记录本轮选出的城市，然后从该城市开始下一轮。
        visited[next] = 1;
        route[i] = next;
        total += minDist;
        current = next;
    }

    // 把从最后一座城市返回起点的距离计入总距离。
    route[n] = start;
    total += dist[current][start];
    printf("最近邻路线：%d", route[0]);
    for (int i = 1; i <= n; ++i) {
        printf(" → %d", route[i]);
    }
    printf("\n路线总距离：%lld\n", total);
}

### 测试用例

依次运行开头的头文件单元、上面的函数定义单元，再运行各组测试。每组先打印预期结果，再调用函数输出实际结果。

In [3]:
{
    printf("=== 测试 1：单城市 ===\n");
    int dist[MAX_CITIES][MAX_CITIES] = {
        {0}
    };
    printf("预期路线：0 → 0\n");
    printf("预期总距离：0\n");
    tsp_nearest_neighbor(dist, 1);
}

=== 测试 1：单城市 ===
预期路线：0 → 0
预期总距离：0
最近邻路线：0 → 0
路线总距离：0


In [4]:
{
    printf("=== 测试 2：距离相同时选择编号较小的城市 ===\n");
    int dist[MAX_CITIES][MAX_CITIES] = {
        {0, 1, 1},
        {1, 0, 2},
        {1, 2, 0}
    };
    printf("预期路线：0 → 1 → 2 → 0\n");
    printf("预期总距离：4\n");
    tsp_nearest_neighbor(dist, 3);
}

=== 测试 2：距离相同时选择编号较小的城市 ===
预期路线：0 → 1 → 2 → 0
预期总距离：4
最近邻路线：0 → 1 → 2 → 0
路线总距离：4


In [5]:
{
    printf("=== 测试 3：四城市反例 ===\n");
    int dist[MAX_CITIES][MAX_CITIES] = {
        {0, 2, 3, 10},
        {2, 0, 5, 8},
        {3, 5, 0, 13},
        {10, 8, 13, 0}
    };
    printf("预期路线：0 → 1 → 2 → 3 → 0\n");
    printf("预期总距离：30（全局最短回路为 26）\n");
    tsp_nearest_neighbor(dist, 4);
}

=== 测试 3：四城市反例 ===
预期路线：0 → 1 → 2 → 3 → 0
预期总距离：30（全局最短回路为 26）
最近邻路线：0 → 1 → 2 → 3 → 0
路线总距离：30


In [6]:
{
    printf("=== 测试 4：单段距离为 INT_MAX ===\n");
    int dist[MAX_CITIES][MAX_CITIES] = {
        {0, INT_MAX},
        {INT_MAX, 0}
    };
    printf("预期路线：0 → 1 → 0\n");
    printf("预期总距离：%lld（两段 INT_MAX）\n", 2LL * INT_MAX);
    tsp_nearest_neighbor(dist, 2);
}

=== 测试 4：单段距离为 INT_MAX ===
预期路线：0 → 1 → 0
预期总距离：4294967294（两段 INT_MAX）
最近邻路线：0 → 1 → 0
路线总距离：4294967294


In [7]:
{
    printf("=== 测试 5：100 座城市的容量上界 ===\n");
    int dist[MAX_CITIES][MAX_CITIES];
    for (int u = 0; u < MAX_CITIES; ++u) {
        for (int v = 0; v < MAX_CITIES; ++v) {
            dist[u][v] = abs(u - v);
        }
    }
    printf("预期路线：0 → 1 → … → 99 → 0\n");
    printf("预期总距离：198\n");
    tsp_nearest_neighbor(dist, MAX_CITIES);
}

=== 测试 5：100 座城市的容量上界 ===
预期路线：0 → 1 → … → 99 → 0
预期总距离：198
最近邻路线：0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 → 15 → 16 → 17 → 18 → 19 → 20 → 21 → 22 → 23 → 24 → 25 → 26 → 27 → 28 → 29 → 30 → 31 → 32 → 33 → 34 → 35 → 36 → 37 → 38 → 39 → 40 → 41 → 42 → 43 → 44 → 45 → 46 → 47 → 48 → 49 → 50 → 51 → 52 → 53 → 54 → 55 → 56 → 57 → 58 → 59 → 60 → 61 → 62 → 63 → 64 → 65 → 66 → 67 → 68 → 69 → 70 → 71 → 72 → 73 → 74 → 75 → 76 → 77 → 78 → 79 → 80 → 81 → 82 → 83 → 84 → 85 → 86 → 87 → 88 → 89 → 90 → 91 → 92 → 93 → 94 → 95 → 96 → 97 → 98 → 99 → 0
路线总距离：198


In [8]:
{
    printf("=== 测试 6：拒绝越界的城市数 ===\n");
    int dist[MAX_CITIES][MAX_CITIES] = {{0}};
    printf("预期结果：n 为 0 或 101 时均提示城市数量超出范围。\n");
    tsp_nearest_neighbor(dist, 0);
    tsp_nearest_neighbor(dist, MAX_CITIES + 1);
}

=== 测试 6：拒绝越界的城市数 ===
预期结果：n 为 0 或 101 时均提示城市数量超出范围。
输入错误：城市数量必须在 1 到 100 之间。
输入错误：城市数量必须在 1 到 100 之间。


### 复杂度分析

以实际城市数 $n$ 为问题规模。按单位代价模型，一次数组访问、比较、赋值或单个城市编号的输出均视为常数时间。

**时间复杂度**

起点已在初始化时记录，外层循环执行 $n-1$ 轮。每轮的内层循环都扫描全部 $n$ 个编号，通过 `visited` 跳过已访问的城市，因此共检查 $n(n-1)$ 个位置。每轮选出城市后的状态更新和最终累加回程距离都只需常数时间，输出路线要遍历 $n+1$ 个位置。按实际城市数分析，初始化访问标记和输出路线各需 $\Theta(n)$ 时间，选城扫描占主导，故总时间复杂度为 **$\Theta(n^2)$**。

**空间复杂度**

按实际城市数分配时，`visited` 需要 $n$ 个位置，`route` 需要 $n+1$ 个位置，其余变量占常数空间。因此额外空间为 **$\Theta(n)$**。输入距离矩阵有 $n^2$ 个元素，计入输入后的总空间为 **$\Theta(n^2)$**。

本 C 实现按固定容量 `MAX_CITIES = 100` 预留数组。若以 $M$ 表示容量，`visited` 和 `route` 分别预留 $M$ 和 $M+1$ 个位置，实际额外空间为 $\Theta(M)$。计入 `visited` 的整数组清零后，时间为 $\Theta(M+n^2)$。测试中距离矩阵按 $M\times M$ 声明，计入该矩阵后的总空间为 $\Theta(M^2)$。

---

## 题目2：汽车加油问题（选做）

**实验目的**

练习用数学模型描述实际问题，并用贪心法求解。

**实验内容**

汽车加满油后最多可行驶 $n$ 公里。沿途有 $k$ 个加油站，给定满油续航 $n$（$n \le 5000$）和这 $k$ 个加油站的位置（$k \le 1000$）。选择沿途停靠加油的站点，使汽车到达终点所需的加油次数最少，并编程计算最少加油次数。

### 算法思路

设满油续航为 `n`，加油站数量为 `k`。为确定行程长度，约定输入还包含终点位置 `D`。起点位置为 `0`，汽车出发时已加满油，途中每次停靠都可重新加满。站点位置和 `D` 均以距起点的公里数表示，加油站位于起点和终点之间。出发时的满油和到达终点均不计入加油次数。

`current` 表示上一次加满油的位置，初始为 `0`。若 `D - current ≤ n`，可以直接到达终点。否则，在满足 `0 < station - current ≤ n` 的加油站中，选择**位置最靠后**的一站加满油，再更新 `current`、停靠记录和加油次数。若没有可达的后续加油站，则无法到达终点。恰好相距 `n` 公里时仍可到达。

先将站点按位置升序排列，再用下标 `i` 从前往后扫描。每轮从第一个尚未扫描的站点开始，沿途记下可达站点，最后一个可达站点就是本轮的选择。选定后，已扫描的站点都位于新的 `current` 处或之前，下一轮可以从 `i` 继续。

**最优性说明**

设当前不能直达终点，且最远的可达站为 `g`。任取一个最优续程方案，它的首个加油站必在 `g` 处或之前。令 `y` 为该方案在 `g` 之后的第一个停靠站，若后面没有停靠站，则令 `y` 为终点。该方案中 `y` 的前驱位置 `p` 满足 `p ≤ g`，且 `y - p ≤ n`，因此从 `g` 到 `y` 也只需一箱油。用在 `g` 的一次加油替换原方案在 `g` 处及之前的所有加油，仍然可行，次数也不增加。因此总能找到一个以 `g` 为下一站的最优方案，剩余路程可重复使用同一规则。

### 伪代码

输入满油续航 `n`、终点位置 `D` 和 `k` 个加油站的位置 `stations`，其中 `1 ≤ n ≤ 5000`、`0 ≤ k ≤ 1000`、`D ≥ 0`，每个站点的位置都严格介于 `0` 和 `D` 之间。

```text
MinRefuels(n, D, stations, k)
    将 stations 按位置从小到大排序
    current ← 0
    i ← 0
    count ← 0
    stops ← 空序列

    当 D - current > n 时
        farthest ← current

        当 i < k 且 stations[i] - current ≤ n 时
            farthest ← stations[i]
            i ← i + 1

        如果 farthest = current
            返回“无法到达”

        current ← farthest
        将 current 追加到 stops
        count ← count + 1

    返回最少加油次数 count 和停靠站序列 stops
```

内层循环先检查 `i < k`，再读取 `stations[i]`。扫描到候选站时只更新 `farthest`，直到本轮选定停靠站才更新 `current`。当 `D ≤ n` 时，外层循环执行零次，返回 `0` 次加油和空的停靠序列。

### 源代码实现

In [9]:
static int compare_station_positions(const void *left, const void *right)
{
    int a = *(const int *)left;
    int b = *(const int *)right;

    return (a > b) - (a < b);
}

int min_refuels(int range, int destination, int stations[], int k, int stops[])
{
    if (k > 1) {
        qsort(stations, (size_t)k, sizeof stations[0],
              compare_station_positions);
    }

    int current = 0;
    int i = 0;
    int count = 0;

    while (destination - current > range) {
        int farthest = current;

        // 从上一次加满油的位置扫描所有可达站点。
        while(i < k && stations[i] <= current + range){
            farthest = stations[i];
            i++;
        }

        if (farthest == current) {
            return -1;
        }

        // 在选中的站点加满油，并记录一次停靠。
        current = farthest;
        stops[count] = current;
        ++count;
    }

    return count;
}

### 测试用例

先运行开头的头文件单元和题目 2 的函数定义单元。每组用例打印预期与实际结果。返回值为 `-1` 表示无法到达终点，此时忽略途中写入的停靠记录。

In [10]:
{
    printf("=== 测试 1：恰好直达终点 ===\n");
    printf("预期返回值：0\n");

    int count = min_refuels(6, 6, NULL, 0, NULL);
    printf("实际返回值：%d\n", count);
}

=== 测试 1：恰好直达终点 ===
预期返回值：0
实际返回值：0


In [11]:
{
    printf("=== 测试 2：站点无序 ===\n");
    int stations[] = {14, 4, 9, 2, 10, 5};
    int stops[6];
    printf("预期返回值：2\n");
    printf("预期停靠位置：5 → 10\n");

    int count = min_refuels(6, 16, stations, 6, stops);
    printf("实际返回值：%d\n", count);
    if (count == 2) {
        printf("实际停靠位置：%d → %d\n", stops[0], stops[1]);
    }
}

=== 测试 2：站点无序 ===
预期返回值：2
预期停靠位置：5 → 10
实际返回值：2
实际停靠位置：5 → 10


In [12]:
{
    printf("=== 测试 3：恰好可达并连续停靠两次 ===\n");
    int stations[] = {4, 8};
    int stops[2];
    printf("预期返回值：2\n");
    printf("预期停靠位置：4 → 8\n");

    int count = min_refuels(4, 12, stations, 2, stops);
    printf("实际返回值：%d\n", count);
    if (count == 2) {
        printf("实际停靠位置：%d → %d\n", stops[0], stops[1]);
    }
}

=== 测试 3：恰好可达并连续停靠两次 ===
预期返回值：2
预期停靠位置：4 → 8
实际返回值：2
实际停靠位置：4 → 8


In [13]:
{
    printf("=== 测试 4：没有加油站且终点超出续航 ===\n");
    printf("预期返回值：-1（无法到达）\n");

    int count = min_refuels(6, 7, NULL, 0, NULL);
    printf("实际返回值：%d\n", count);
}

=== 测试 4：没有加油站且终点超出续航 ===
预期返回值：-1（无法到达）
实际返回值：-1


In [14]:
{
    printf("=== 测试 5：途中出现超出续航的路段 ===\n");
    int stations[] = {4, 10};
    int stops[2];
    printf("预期返回值：-1（无法到达）\n");

    int count = min_refuels(4, 13, stations, 2, stops);
    printf("实际返回值：%d\n", count);
}

=== 测试 5：途中出现超出续航的路段 ===
预期返回值：-1（无法到达）
实际返回值：-1


In [15]:
{
    printf("=== 测试 6：重复的加油站位置 ===\n");
    int stations[] = {6, 6};
    int stops[2];
    printf("预期返回值：1\n");
    printf("预期停靠位置：6\n");

    int count = min_refuels(6, 12, stations, 2, stops);
    printf("实际返回值：%d\n", count);
    if (count == 1) {
        printf("实际停靠位置：%d\n", stops[0]);
    }
}

=== 测试 6：重复的加油站位置 ===
预期返回值：1
预期停靠位置：6
实际返回值：1
实际停靠位置：6


### 复杂度分析

以加油站数量 $k$ 为问题规模。按单位代价模型，整数运算、比较和数组访问均视为常数时间。

**时间复杂度**

- **排序阶段**。当 $k>1$ 时，函数调用 `qsort` 将 `stations` 按位置升序排列，每次比较两个站点的代价为常数。设排序的时间开销为 $T_{\mathrm{sort}}(k)$。C 标准未规定 `qsort` 的排序算法或复杂度，具体开销取决于所用的标准库实现。
- **贪心扫描阶段**。下标 `i` 在外层循环之外初始化，并且只向前移动，所以内层循环体在整个行程中最多执行 $k$ 次。每次成功停靠至少扫描一个新站点，每轮结束时最多还有一次终止检查。即使途中不可达，也只会额外执行一轮失败检查。因此扫描的最坏时间为 **$\Theta(k)$**。若能从起点直达终点，扫描阶段只需 $\Theta(1)$ 时间。

总时间为排序开销加扫描开销，即 $T_{\mathrm{sort}}(k)+O(k+1)$。如果使用最坏时间为 $O(k\log k)$ 的排序实现，整体最坏时间也是 **$O(k\log k)$**。当前代码即使收到已经有序的站点，在 $k>1$ 时仍会调用 `qsort`。

**空间复杂度**

- **排序阶段**。函数直接排序传入的 `stations`，没有另外复制站点数组。设 `qsort` 内部使用的辅助空间为 $S_{\mathrm{sort}}(k)$，其大小取决于具体实现。
- **贪心扫描阶段**。只用 `current`、`i`、`count` 和 `farthest` 等少量变量，扣除输入和输出数组后，额外空间为 **$\Theta(1)$**。调用方为 `stops` 预留 $k$ 个位置，输出缓冲区占 $\Theta(k)$ 空间。

不计输入和输出数组，整个函数的额外空间为 $O(1+S_{\mathrm{sort}}(k))$。若把输入站点和输出停靠记录都计入，总空间为 $O(k+1+S_{\mathrm{sort}}(k))$。